# Stock Price Data Exploration and Preprocessing

This notebook demonstrates data loading, exploration, and preprocessing for stock price prediction using LSTM models with uncertainty quantification.

## Objectives
1. Load stock price data from Yahoo Finance
2. Explore data characteristics and patterns
3. Calculate technical indicators
4. Preprocess data for LSTM models
5. Prepare train/validation/test splits

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from data.data_loader import StockDataLoader, load_sample_data
from data.preprocessor import StockDataPreprocessor, create_lagged_features, create_rolling_features
from utils.config import get_config
from utils.visualization import StockVisualization

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## 1. Configuration and Setup

In [ ]:
# Load configuration
config = get_config()

print("Configuration:")
print(f"Stock symbols: {config.data.symbols}")
print(f"Date range: {config.data.start_date} to {config.data.end_date}")
print(f"Features: {config.data.features}")
print(f"Sequence length: {config.data.sequence_length}")
print(f"Prediction horizon: {config.data.prediction_horizon}")

## 2. Data Loading

In [ ]:
# Initialize data loader
data_loader = StockDataLoader(
    symbols=config.data.symbols,
    start_date=config.data.start_date,
    end_date=config.data.end_date
)

# Load data
print("Loading stock data...")
stock_data = data_loader.load_data(features=config.data.features)

# Display basic information
for symbol, data in stock_data.items():
    print(f"\n{symbol}:")
    print(f"  Shape: {data.shape}")
    print(f"  Date range: {data.index[0]} to {data.index[-1]}")
    print(f"  Columns: {list(data.columns)}")

## 3. Data Exploration

In [ ]:
# Select a primary stock for detailed analysis
primary_symbol = config.data.symbols[0]  # Use first symbol
primary_data = stock_data[primary_symbol]

print(f"Analyzing {primary_symbol} in detail...")
print(f"\nBasic Statistics:")
print(primary_data.describe())

In [ ]:
# Check for missing values
print("Missing values:")
missing_values = primary_data.isnull().sum()
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("No missing values found!")

In [ ]:
# Initialize visualization
visualizer = StockVisualization(figsize=(15, 8))

# Plot stock price data
fig = visualizer.plot_stock_data(
    primary_data,
    columns=['Close', 'Open', 'High', 'Low'],
    title=f"{primary_symbol} Stock Price Data"
)
plt.show()

In [ ]:
# Plot technical indicators
fig = visualizer.plot_technical_indicators(primary_data)
plt.show()

## 4. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = primary_data.corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, 
            mask=mask,
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            fmt='.2f',
            cbar_kws={"shrink": .8})
plt.title(f'{primary_symbol} - Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze correlation with close price
close_correlations = correlation_matrix['Close'].sort_values(ascending=False)
print("Correlations with Close price:")
print(close_correlations)

## 5. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = StockDataPreprocessor(
    sequence_length=config.data.sequence_length,
    prediction_horizon=config.data.prediction_horizon,
    scaler_type="standard"
)

# Select features for modeling (excluding target)
feature_columns = [col for col in primary_data.columns if col != 'Close']
target_column = 'Close'

print(f"Selected features: {feature_columns}")
print(f"Target column: {target_column}")

In [ ]:
# Split data chronologically
train_size = int(len(primary_data) * config.data.train_ratio)
val_size = int(len(primary_data) * config.data.val_ratio)

train_data = primary_data.iloc[:train_size]
val_data = primary_data.iloc[train_size:train_size + val_size]
test_data = primary_data.iloc[train_size + val_size:]

print(f"Training data: {len(train_data)} samples")
print(f"Validation data: {len(val_data)} samples")
print(f"Test data: {len(test_data)} samples")
print(f"Total: {len(primary_data)} samples")

In [ ]:
# Fit scalers on training data
print("Fitting scalers on training data...")
preprocessor.fit_scalers(train_data, target_column=target_column)

# Transform data into sequences
print("Creating sequences...")
X_train, y_train = preprocessor.transform_data(train_data, target_column=target_column)
X_val, y_val = preprocessor.transform_data(val_data, target_column=target_column)
X_test, y_test = preprocessor.transform_data(test_data, target_column=target_column)

print(f"\nSequence shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

## 6. Data Analysis and Insights

In [ ]:
# Analyze price distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Price distribution
axes[0].hist(primary_data['Close'], bins=50, alpha=0.7, edgecolor='black')
axes[0].set_title('Close Price Distribution')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Daily returns distribution
daily_returns = primary_data['Close'].pct_change().dropna()
axes[1].hist(daily_returns, bins=50, alpha=0.7, edgecolor='black')
axes[1].set_title('Daily Returns Distribution')
axes[1].set_xlabel('Daily Return')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

# Volume distribution
axes[2].hist(primary_data['Volume'], bins=50, alpha=0.7, edgecolor='black')
axes[2].set_title('Volume Distribution')
axes[2].set_xlabel('Volume')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print(f"Daily returns statistics:")
print(f"Mean: {daily_returns.mean():.4f}")
print(f"Std: {daily_returns.std():.4f}")
print(f"Skewness: {daily_returns.skew():.4f}")
print(f"Kurtosis: {daily_returns.kurtosis():.4f}")

In [ ]:
# Analyze seasonality patterns
primary_data_copy = primary_data.copy()
primary_data_copy['Year'] = primary_data_copy.index.year
primary_data_copy['Month'] = primary_data_copy.index.month
primary_data_copy['DayOfWeek'] = primary_data_copy.index.dayofweek
primary_data_copy['DayOfYear'] = primary_data_copy.index.dayofyear

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Monthly patterns
monthly_returns = primary_data_copy.groupby('Month')['Close'].pct_change().groupby(primary_data_copy['Month']).mean()
axes[0, 0].bar(monthly_returns.index, monthly_returns.values)
axes[0, 0].set_title('Average Monthly Returns')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Average Return')
axes[0, 0].grid(True, alpha=0.3)

# Day of week patterns
dow_returns = primary_data_copy.groupby('DayOfWeek')['Close'].pct_change().groupby(primary_data_copy['DayOfWeek']).mean()
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
axes[0, 1].bar(range(len(dow_returns)), dow_returns.values)
axes[0, 1].set_title('Average Daily Returns by Day of Week')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Average Return')
axes[0, 1].set_xticks(range(len(dow_names)))
axes[0, 1].set_xticklabels(dow_names)
axes[0, 1].grid(True, alpha=0.3)

# Yearly patterns
yearly_returns = primary_data_copy.groupby('Year')['Close'].mean()
axes[1, 0].plot(yearly_returns.index, yearly_returns.values, marker='o')
axes[1, 0].set_title('Average Yearly Prices')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Average Price')
axes[1, 0].grid(True, alpha=0.3)

# Volatility over time
rolling_vol = primary_data['Close'].pct_change().rolling(window=30).std()
axes[1, 1].plot(rolling_vol.index, rolling_vol.values)
axes[1, 1].set_title('30-Day Rolling Volatility')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Volatility')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Get feature importance data
feature_info = preprocessor.get_feature_importance_data(primary_data)

# Create feature importance plot based on correlation with target
correlations = []
feature_names = []

for feature, stats in feature_info['feature_stats'].items():
    if stats['correlation_with_close'] is not None and feature != 'Close':
        correlations.append(abs(stats['correlation_with_close']))
        feature_names.append(feature)

# Sort by correlation
sorted_indices = np.argsort(correlations)[::-1]
sorted_correlations = [correlations[i] for i in sorted_indices]
sorted_features = [feature_names[i] for i in sorted_indices]

# Plot feature importance
plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(sorted_features)), sorted_correlations)
plt.yticks(range(len(sorted_features)), sorted_features)
plt.xlabel('Absolute Correlation with Close Price')
plt.title('Feature Importance (Correlation with Target)')
plt.grid(True, alpha=0.3)

# Color bars by importance
for i, bar in enumerate(bars):
    bar.set_color(plt.cm.viridis(i / len(bars)))

plt.tight_layout()
plt.show()

print("Top 10 most correlated features:")
for i, (feature, corr) in enumerate(zip(sorted_features[:10], sorted_correlations[:10])):
    print(f"{i+1:2d}. {feature:<20}: {corr:.4f}")

## 8. Data Quality Summary

In [ ]:
print("DATA QUALITY SUMMARY")
print("=" * 50)
print(f"Dataset: {primary_symbol}")
print(f"Time period: {primary_data.index[0].strftime('%Y-%m-%d')} to {primary_data.index[-1].strftime('%Y-%m-%d')}")
print(f"Total samples: {len(primary_data):,}")
print(f"Number of features: {len(primary_data.columns)}")
print(f"Missing values: {primary_data.isnull().sum().sum()}")

print(f"\nTRAINING DATA SPLIT:")
print(f"Training sequences: {len(X_train):,}")
print(f"Validation sequences: {len(X_val):,}")
print(f"Test sequences: {len(X_test):,}")
print(f"Sequence length: {config.data.sequence_length}")
print(f"Number of features per timestep: {X_train.shape[2]}")

print(f"\nPRICE STATISTICS:")
print(f"Price range: ${primary_data['Close'].min():.2f} - ${primary_data['Close'].max():.2f}")
print(f"Average price: ${primary_data['Close'].mean():.2f}")
print(f"Price volatility (std): ${primary_data['Close'].std():.2f}")

daily_returns = primary_data['Close'].pct_change().dropna()
print(f"\nRETURNS STATISTICS:")
print(f"Average daily return: {daily_returns.mean()*100:.4f}%")
print(f"Daily volatility: {daily_returns.std()*100:.4f}%")
print(f"Annualized volatility: {daily_returns.std()*np.sqrt(252)*100:.2f}%")
print(f"Sharpe ratio: {daily_returns.mean()/daily_returns.std()*np.sqrt(252):.4f}")

print(f"\nData is ready for LSTM model training!")

## 9. Save Preprocessed Data

In [ ]:
# Save preprocessed data
import pickle
import os

# Create data directory if it doesn't exist
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)

# Save processed sequences
processed_data = {
    'X_train': X_train,
    'y_train': y_train,
    'X_val': X_val,
    'y_val': y_val,
    'X_test': X_test,
    'y_test': y_test,
    'preprocessor': preprocessor,
    'config': config,
    'symbol': primary_symbol,
    'raw_data': primary_data
}

with open(f'{data_dir}/processed_data_{primary_symbol}.pkl', 'wb') as f:
    pickle.dump(processed_data, f)

print(f"Preprocessed data saved to {data_dir}/processed_data_{primary_symbol}.pkl")
print("\nData exploration and preprocessing completed successfully!")